# 04. SAST for ML Pipelines

## Background

ML pipelines have unique vulnerability patterns that general-purpose SAST tools miss: unsafe model loading, prompt injection from training data, API key leakage in notebooks, and data poisoning through unsanitized inputs. This notebook builds an ML-specific SAST scanner that catches these patterns.

## Why This Matters

Automated security scanning in CI prevents vulnerabilities from reaching production. Each scan type catches a different class of vulnerability — layering them provides defense in depth.

In [ ]:
import ast, re, json
from dataclasses import dataclass
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print(f'04. SAST for ML Pipelines ready')


## ML-Specific Vulnerability Scanner

In [ ]:
import ast, re, json
from dataclasses import dataclass, field

@dataclass
class MLFinding:
    rule_id:   str
    severity:  str
    category:  str
    message:   str
    file:      str
    line:      int
    cwe:       str = ''
    fix:       str = ''


ML_SAST_RULES = [
    # Model loading
    ('ML001','CRITICAL','model_loading',  'torch.load() without weights_only=True executes arbitrary code',
     r'torch\.load\s*\(',  'torch.load(..., weights_only=True)', 'CWE-502'),
    ('ML002','HIGH',    'model_loading',  'pickle.load/loads on model file',
     r'pickle\.(load|loads)\s*\(', 'Use safetensors or onnx format', 'CWE-502'),
    ('ML003','HIGH',    'model_loading',  'joblib.load from untrusted path',
     r'joblib\.load\s*\(', 'Hash-verify before loading; use MLflow model registry', 'CWE-502'),
    # Prompt injection surfaces
    ('PI001','HIGH',    'prompt_injection','User input interpolated directly into prompt',
     r'f["\'].*{.*user.*(input|query|message).*}', 'Sanitize user input; use system/user message roles', 'CWE-74'),
    ('PI002','MEDIUM',  'prompt_injection','Document content interpolated into prompt without sanitization',
     r'f["\'].*{.*(doc|text|content|chunk).*}', 'Strip control characters; use content safety filters', 'CWE-74'),
    # Credential leakage
    ('CR001','CRITICAL','credentials',    'Hardcoded API key in ML code',
     r'(api_key|openai|anthropic)\s*=\s*["\'][A-Za-z0-9_\-]{20,}', 'Use os.environ or secrets manager', 'CWE-798'),
    # Data handling
    ('DH001','MEDIUM',  'data_handling',  'DataFrame from_csv without dtype validation',
     r'pd\.read_csv\s*\((?!.*dtype)', 'Specify dtypes explicitly to prevent type confusion', 'CWE-704'),
    ('DH002','HIGH',    'data_handling',  'eval() on DataFrame column — arbitrary code execution',
     r'\.eval\s*\(', 'Use vectorized pandas operations instead', 'CWE-94'),
]

def scan_ml_code(code, filename='pipeline.py'):
    findings = []
    for lineno, line in enumerate(code.split('\n'), 1):
        for rule_id, sev, cat, msg, pattern, fix, cwe in ML_SAST_RULES:
            if re.search(pattern, line, re.IGNORECASE):
                findings.append(MLFinding(rule_id, sev, cat, msg, filename, lineno, cwe, fix))
    return findings


VULN_ML = """
import torch, pickle, joblib, pandas as pd
from openai import OpenAI

api_key = 'sk-proj-abc123XYZdef456'
client  = OpenAI(api_key=api_key)

model = torch.load('model.pt')  # unsafe
clf   = joblib.load('classifier.pkl')  # unsafe

def build_prompt(user_query, doc_content):
    prompt = f'Answer based on: {doc_content}. Question: {user_query}'
    return prompt

df = pd.read_csv('data.csv')
result = df.eval(user_formula)  # arbitrary code!
"""

findings = scan_ml_code(VULN_ML)
for f in findings:
    print(f'[{f.severity:8s}] {f.rule_id} {f.category:<18} line {f.line}: {f.message[:60]}')
    print(f'               Fix: {f.fix}')


## Security Report Summary

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def security_report(findings):
    by_sev = Counter(f.severity for f in findings)
    by_cat = Counter(f.category for f in findings)
    print(f'Total findings: {len(findings)}')
    print(f'By severity:  {dict(by_sev)}')
    print(f'By category:  {dict(by_cat)}')
    risk = sum({'CRITICAL':10,'HIGH':5,'MEDIUM':2,'LOW':1}.get(f.severity,0) for f in findings)
    print(f'Risk score:   {risk}')
    return by_sev, by_cat

by_sev, by_cat = security_report(findings)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_colors = {'CRITICAL':'#B71C1C','HIGH':'#F44336','MEDIUM':'#FF9800','LOW':'#FFC107'}
axes[0].bar(by_sev.keys(), by_sev.values(),
            color=[ax_colors.get(k,'gray') for k in by_sev.keys()])
axes[0].set_title('ML SAST — Findings by Severity')
axes[1].bar(by_cat.keys(), by_cat.values(), color='steelblue', alpha=0.8)
axes[1].set_title('ML SAST — Findings by Category')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('04_ml_sast.png', dpi=110, bbox_inches='tight')
plt.show()
